# Stock Research Agent with Grounded Base Rates

Build a Claude agent that **never hallucinates forward-return statistics**.
Instead of guessing what usually happens after a pattern, the agent calls a
single tool that returns real historical conditional distributions with
sample size and survivorship flag.

This notebook demonstrates:
1. The hallucination failure mode (same prompt, no tool)
2. The grounded-answer pattern (with `get_cohort_distribution` tool)
3. Progressive refinement via `explain_cohort_filters` + `refine_cohort_with_filters`

**Prereqs**
```bash
pip install anthropic requests
export ANTHROPIC_API_KEY=sk-ant-...
export CHART_LIBRARY_KEY=cl_...     # free key at chartlibrary.io/developers
```

In [ ]:
import json
import os

import anthropic
import requests

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

CHART_BASE = "https://chartlibrary.io"
CHART_KEY = os.environ["CHART_LIBRARY_KEY"]
CHART_HEADERS = {
    "Authorization": f"Bearer {CHART_KEY}",
    "Content-Type": "application/json",
}
MODEL = "claude-sonnet-4-5"

## 1. The failure mode: hallucinated base rates

Ask Claude a specific forward-return question with no tools. The answer
will sound authoritative but contains invented statistics.

In [ ]:
user_question = (
    "For NVDA on 2024-06-18 — a high-momentum breakout setup — what does "
    "the 5-day forward return distribution usually look like, conditional "
    "on a similar VIX regime and same sector (technology)? I need "
    "percentiles and a sample size."
)

resp = client.messages.create(
    model=MODEL,
    max_tokens=700,
    messages=[{"role": "user", "content": user_question}],
)
print("─── UNGROUNDED ANSWER (no tool) ───")
print(resp.content[0].text)

**Observe:** Claude produces a plausible-sounding answer. Sample size and
percentiles are invented — they look real because they're formatted like
real statistics. A user acting on this is acting on a guess.

## 2. The fix: a tool that returns real distributions

Chart Library's `/api/v1/cohort` endpoint embeds the anchor chart,
filters the historical corpus by regime/sector/liquidity/event filters,
and returns percentile distributions of forward returns, MAE, MFE, and
realized vol — with sample size and survivorship flag.

We wire it into Claude as a single tool:

In [ ]:
COHORT_TOOL = {
    "name": "get_cohort_distribution",
    "description": (
        "Return the historical forward-return distribution for a chart "
        "pattern, filtered by regime/sector/liquidity. Every response includes "
        "sample size, MAE/MFE (max adverse/favorable excursion) percentiles, "
        "realized-vol percentiles, and a survivorship flag (how many delisted "
        "names are in the cohort). Call this INSTEAD of guessing forward-"
        "return statistics."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "symbol": {"type": "string", "description": "Ticker (e.g. 'NVDA')"},
            "date":   {"type": "string", "description": "ISO date of the setup (YYYY-MM-DD)"},
            "same_sector":     {"type": "boolean", "default": False,
                                "description": "Keep only matches from the same sector as the anchor"},
            "same_vix_bucket": {"type": "boolean", "default": False,
                                "description": "Keep only matches whose VIX regime is within ±0.15 of anchor"},
            "same_trend":      {"type": "boolean", "default": False,
                                "description": "Keep only matches whose SPY 20d trend percentile is within ±0.15 of anchor"},
            "horizons":        {"type": "array", "items": {"type": "integer"},
                                "default": [5, 10],
                                "description": "Forward horizons in trading days"},
            "top_k":           {"type": "integer", "default": 500,
                                "description": "Cohort size used for stats (larger = tighter CIs)"},
        },
        "required": ["symbol", "date"],
    },
}

REFINE_TOOL = {
    "name": "refine_cohort_with_filters",
    "description": (
        "Narrow a previously-returned cohort with additional filters. "
        "Sub-second because there's no kNN re-run. Use this AFTER "
        "get_cohort_distribution to add more specific filters."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "cohort_id":       {"type": "string", "description": "ID from a prior get_cohort_distribution response"},
            "same_vix_bucket": {"type": "boolean", "default": False},
            "same_trend":      {"type": "boolean", "default": False},
        },
        "required": ["cohort_id"],
    },
}

EXPLAIN_TOOL = {
    "name": "explain_cohort_filters",
    "description": (
        "For a previously-returned cohort, rank which additional filter "
        "would shift the return distribution most. Tells the agent which "
        "dimension is actually driving outcomes for this specific setup."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "cohort_id": {"type": "string"},
            "horizon":   {"type": "integer", "default": 5},
        },
        "required": ["cohort_id"],
    },
}

TOOLS = [COHORT_TOOL, REFINE_TOOL, EXPLAIN_TOOL]


def call_cohort(**args):
    filters = {}
    if args.get("same_sector"):
        filters["sector"] = "same_as_anchor"
    regime = {}
    if args.get("same_vix_bucket"): regime["same_vix_bucket"] = True
    if args.get("same_trend"):      regime["same_trend"] = True
    if regime: filters["regime"] = regime
    body = {
        "anchor": {"symbol": args["symbol"], "date": args["date"]},
        "filters": filters,
        "horizons": args.get("horizons", [5, 10]),
        "top_k": args.get("top_k", 500),
        "include_path_stats": True,
    }
    r = requests.post(f"{CHART_BASE}/api/v1/cohort", headers=CHART_HEADERS, json=body, timeout=30)
    r.raise_for_status()
    return r.json()


def call_refine(**args):
    cohort_id = args["cohort_id"]
    extra = {}
    regime = {}
    if args.get("same_vix_bucket"): regime["same_vix_bucket"] = True
    if args.get("same_trend"):      regime["same_trend"] = True
    if regime: extra["regime"] = regime
    r = requests.post(
        f"{CHART_BASE}/api/v1/cohort/{cohort_id}/filter",
        headers=CHART_HEADERS,
        json={"extra_filters": extra, "include_path_stats": True},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()


def call_explain(**args):
    r = requests.get(
        f"{CHART_BASE}/api/v1/cohort/{args['cohort_id']}/explain",
        headers=CHART_HEADERS,
        params={"horizon": args.get("horizon", 5)},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()


TOOL_FNS = {
    "get_cohort_distribution": call_cohort,
    "refine_cohort_with_filters": call_refine,
    "explain_cohort_filters": call_explain,
}

## 3. The agent loop

System prompt forces tool use for any forward-return claim. Agent loop
runs until Claude stops requesting tools.

In [ ]:
SYSTEM_PROMPT = """You are a stock-research assistant that NEVER invents forward-return statistics.

Rules you MUST follow:
1. If the user asks about forward returns, hit rates, drawdowns, or pattern outcomes, call `get_cohort_distribution` FIRST.
2. Quote the sample size in your answer (e.g. "based on n=491 historical analogs").
3. Disclose the survivorship flag — say how many delisted names were in the cohort.
4. If you want to narrow the cohort, use `explain_cohort_filters` to see which filter moves the distribution most, then `refine_cohort_with_filters` to fork it.
5. Never quote a percentile you didn't see in tool output. If tool output is absent, say so plainly.
"""


def run_agent(user_question: str, max_iters: int = 5) -> str:
    messages = [{"role": "user", "content": user_question}]
    for _ in range(max_iters):
        resp = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if getattr(b, "type", None) == "text")

        tool_results = []
        for block in resp.content:
            if getattr(block, "type", None) != "tool_use":
                continue
            print(f"[tool] {block.name}({json.dumps(block.input)[:200]})")
            fn = TOOL_FNS[block.name]
            try:
                out = fn(**block.input)
                content = json.dumps(out, default=str)[:8000]
            except Exception as e:
                content = json.dumps({"error": str(e)})
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": content,
            })
        messages.append({"role": "user", "content": tool_results})
    return "[max iterations]"

## 4. Grounded answer for the same question

In [ ]:
answer = run_agent(user_question)
print("\n─── GROUNDED ANSWER (with tools) ───")
print(answer)

**Observe:** every percentile, hit rate, and sample-size claim in the
answer comes from the tool call above. The survivorship number is real.
If NVDA's specific regime has only 50 clean analogs, the agent says so
instead of inventing a larger round number.

## 5. Progressive refinement (the edge-mining loop)

A single cohort call is often the start, not the end. For a sharp answer,
the agent should ask which filter moves the distribution most and narrow
based on the answer.

In [ ]:
refinement_question = (
    "For AAPL on 2024-06-18, run an initial cohort, then use "
    "explain_cohort_filters to see which additional filter shifts the 5d "
    "distribution most, then refine with that filter. Report the baseline "
    "vs refined p50 return and above_entry hit rate, and explain what "
    "conditional structure that reveals about the setup."
)
print(run_agent(refinement_question))

## What we built

- A Claude agent wired to a **single retrieval primitive** that returns
  real conditional distributions
- A **system prompt** that forbids fabricating forward-return statistics
- A **refinement loop** (explain → refine) that lets the agent discover
  conditional structure rather than pattern-match to a canned base rate

The pattern is domain-independent. Any finance/research agent that answers
"what usually happens" questions needs exactly this shape: retrieval tool
returning conditional distributions + system prompt requiring its use.

## Learn more

- [chartlibrary.io/developers](https://chartlibrary.io/developers) —
  free API key, docs, pricing (Sandbox free, Agent $299/mo)
- [chartlibrary-mcp on PyPI](https://pypi.org/project/chartlibrary-mcp/) —
  MCP server exposing these tools directly to Claude Desktop and other
  MCP-compatible hosts
- [Chart Library blog: How to Build a Stock-Research Agent That Doesn't Hallucinate](https://chartlibrary.io/blog/how-to-build-a-stock-agent-that-doesnt-hallucinate)